# 05 · CoRE MOF 2019: the 160 K discharge dataset and models
Reproduces report section 6.5:
1. Download CoRE MOF 2019 (12,023 structures, MOFSimplify repository copy); metadata and fast pore sizes.
2. 10-second Widom runs at 160 K on 1,069 MOFs (1,000 stage-A picks + 69 earlier ones).
3. 160 K/5 bar GCMC on the 500 selected MOFs + 58 library MOFs; 77 K/100 bar on 30 test MOFs.
4. Models for the 160 K term (`train_160.py`) and the physics-informed Langmuir head (`pi_head.py`).

The selection lists from the original project (`core_stageA.csv`, `core_stageB.csv`) are used so the same MOFs are simulated; set `RESELECT = True` to redo the selection from your own Widom results.

**Runtime: about 8–11 hours on 4 cores** (1,657 runs); resumable. **Attach:** the `mofh2_pipeline` dataset.

### Settings
* `USE_REFERENCE_UPSTREAM`: fill results this notebook needs from earlier stages with the original project's runs (packaged in `reference_runs.tgz`) when you have not attached your own earlier notebook outputs. Lets every notebook run on its own.
* `RECOMPUTE_THIS_STAGE`: re-run this notebook's own simulations. Set `False` to only redo the analysis on the reference runs.
* `MAX_HOURS`: stop launching new simulations after this many hours (Kaggle sessions end at 12 h). Every queue is resumable: save the output, attach it to a new session, run again.

In [ ]:
import os, glob, subprocess
PKG = os.path.dirname(glob.glob('/kaggle/input/**/mofh2_pipeline/setup_env.sh', recursive=True)[0])
os.environ['WORK'] = WORK = '/kaggle/working/mofh2'
os.environ['NPROC'] = str(os.cpu_count())
USE_REFERENCE_UPSTREAM = True
RECOMPUTE_THIS_STAGE = True
MAX_HOURS = 11.0
STAGE_QUEUES = ['05_core2019_160K']
print('package:', PKG, '| cores:', os.cpu_count())

### Setup: build RASPA2 and Zeo++, install packages (about 5–8 minutes)

In [ ]:
!bash $PKG/setup_env.sh

### Restore results from earlier stages

In [ ]:
%cd $WORK
!python stage_io.py restore {'--reference' if USE_REFERENCE_UPSTREAM else ''}
if RECOMPUTE_THIS_STAGE:
    for q in STAGE_QUEUES:
        !python stage_io.py forget queues/{q}.txt

### 1 · CoRE MOF 2019 structures and fast metadata

In [ ]:
if not os.path.isdir(f'{WORK}/coredl/MOFSimplify/CoRE2019'):
    !mkdir -p coredl && cd coredl && git clone -q --depth 1 --filter=blob:none --no-checkout https://github.com/hjkgrp/MOFSimplify && cd MOFSimplify && git sparse-checkout set --no-cone 'CoRE2019/*' && git checkout -q HEAD
!ls coredl/MOFSimplify/CoRE2019 | wc -l
!python core_meta.py coredl/MOFSimplify/CoRE2019 core_meta.csv && python core_res.py

In [ ]:
RESELECT = False
if RESELECT:
    !python select_core.py   # stage A (1,000 MOFs); then run the Widom part of the queue, then select_core_B.py, then rebuild queues

### Run simulations: `05_core2019_160K` (1,657 runs: Widom, then 160 K GCMC, then 77 K test runs)

In [ ]:
!bash rq_any.sh queues/05_core2019_160K.txt {MAX_HOURS}

### 4 · Featurise the 500 and train

In [ ]:
!python featurise_core.py && python featurise_real.py && python featurise_t130.py && python train_160.py

In [ ]:
!python pi_head.py
!PI_BULK=1 PI_KS=0,100,9999 python pi_head.py

### Compare with the original project and save outputs
The comparison prints mean and maximum differences between your reproduced numbers and the original ones. GCMC is stochastic, so expect small differences (typically under 1 g/L at 77 K/100 bar).

In [ ]:
!python compare_reference.py $PKG train_160_results.csv train_160_doe.csv pi_head_results.csv

In [ ]:
!python stage_io.py save stage05_outputs
# Attach /kaggle/working/stage05_outputs.tgz (via 'Save Version' output) as input to later notebooks.